# Water as an Acid and a Base

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/temp/12-acids-bases.ipynb)
:::


In [1]:
####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB:
    if not Path("styles/tufte.mplstyle").exists():
        !wget -q -N -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
        !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py
    plt.style.use("styles/tufte.mplstyle")
    nproc = 2          # Colab gives you 2 cores
else:
    sys.path.insert(0, "..")                    # psi4_helpers.py is one folder up
    plt.style.use("../styles/tufte.mplstyle")
    nproc = 4          # house limit: 4 threads

import psi4_helpers as ph

# All Psi4 .out and .log files go into the output folder
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Resources: house limits are 4 threads and 4 GB
psi4.set_num_threads(nproc)
_ = psi4.set_memory("2 GB")


  Threads set to 4 by Python driver.



  Memory set to   1.863 GiB by Python driver.


Water is both a base and an acid. In the gas phase, with no solvent to help, we can calculate exactly how much energy it takes to add or remove a proton:

$$
\mathrm{H_3O^+ \rightarrow H_2O + H^+} \qquad \Delta H = \text{proton affinity (PA) of water}
$$
$$
\mathrm{H_2O \rightarrow HO^- + H^+} \qquad \Delta H = \text{gas-phase acidity } \Delta H_{\mathrm{acid}}
$$

Experiment (NIST WebBook): PA(H₂O) = **691 kJ/mol** (gas basicity $\Delta G$ = 660 kJ/mol); $\Delta H_{\mathrm{acid}}$(H₂O) = **1633 kJ/mol** ($\Delta G_{\mathrm{acid}}$ = 1607 kJ/mol).

## What You Will Learn
- How to set the **charge** of a molecule in *Psi4*.
- How to calculate reaction enthalpies and free energies from optimizations and frequency calculations.
- How to handle the **proton**, which has no electrons.
- Why **anions need diffuse basis functions**.

In [2]:
psi4.set_output_file("output/12-acids-bases.out", False);
h2kJ = psi4.constants.hartree2kJmol
R = 8.314462618e-3        # kJ/(mol K)
T = 298.15

## 1. The Three Species

The first line of each geometry gives the **charge** and **multiplicity**: `1 1` for the cation, `-1 1` for the anion. All three are closed-shell singlets.

```{note} Psi4 settings
:class: dropdown
`b3lyp` with three basis sets
: `6-31g*` (no diffuse functions), `6-31+g*` (adds diffuse *sp* functions on O), and `aug-cc-pvtz` (large, with diffuse functions on every atom). *Alternatives:* `mp2/aug-cc-pvtz` or a CCSD(T) single point on the B3LYP structures (written CCSD(T)/aug-cc-pVTZ//B3LYP/6-31+G*) for higher accuracy; `def2-tzvpd` is a good diffuse basis in the Ahlrichs family.

`reference: "rhf"` (default)
: All three species are closed shell, so restricted HF/Kohn–Sham is correct. Charge alone does not require UHF; an odd number of electrons does.

Why a frequency calculation for each species?
: The reaction enthalpy needs the ZPVE and thermal corrections (see the *Thermochemistry* notebook). The proton itself has no electrons, no vibrations and no rotations: its electronic energy is exactly zero and its enthalpy is just translation plus $PV$, $H(\mathrm{H^+}) = \tfrac{5}{2}RT = 6.2$ kJ/mol. Its entropy at 1 atm is 108.9 J/(mol K), so $G(\mathrm{H^+}) = -26.3$ kJ/mol.
```

In [3]:
species = {
    "H2O": """
        0 1
        O
        H 1 0.96
        H 1 0.96 2 104.5
    """,
    "H3O+": """
        1 1
        O    0.000   0.000   0.100
        H    0.000   0.950  -0.250
        H    0.823  -0.475  -0.250
        H   -0.823  -0.475  -0.250
    """,
    "OH-": """
        -1 1
        O
        H 1 0.97
    """,
}

H_proton = 2.5 * R * T                       # kJ/mol
G_proton = H_proton - T * 108.9e-3           # kJ/mol

## 2. Optimize and Calculate Frequencies for Each Basis Set

In [4]:
import time

basis_sets = ["6-31g*", "6-31+g*", "aug-cc-pvtz"]
rows = []
for basis in basis_sets:
    psi4.set_options({"basis": basis, "scf_type": "df"})
    for name, geom in species.items():
        t0 = time.perf_counter()
        mol = psi4.geometry(geom)
        psi4.optimize("b3lyp", molecule=mol)
        E, wfn = psi4.frequency("b3lyp", molecule=mol, return_wfn=True)
        H, G = psi4.variable("ENTHALPY"), psi4.variable("GIBBS FREE ENERGY")
        E_sp, sp_wfn = psi4.energy("b3lyp", molecule=mol, return_wfn=True)       # for the orbital energies
        homo = sp_wfn.epsilon_a_subset("AO", "OCC").np[-1] * psi4.constants.hartree2ev
        rows.append({"basis": basis, "species": name,
                     "H (hartree)": H,
                     "G (hartree)": G,
                     "HOMO (eV)": homo,
                     "lowest freq (cm-1)": wfn.frequencies().np.min(),
                     "time (s)": time.perf_counter() - t0})

data = pd.DataFrame(rows)
data.round(4)

Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


,basis,species,H (hartree),G (hartree),HOMO (eV),lowest freq (cm-1),time (s)
0,6-31g*,H2O,-76.3840,-76.4055,-7.9241,1713.1897,6.5455
1,6-31g*,H3O+,-76.6509,-76.6739,-20.1938,903.2704,11.1068
2,6-31g*,OH-,-75.7097,-75.7293,4.3755,3417.7215,3.1342
3,6-31+g*,H2O,-76.3977,-76.4191,-8.6875,1662.2644,5.9707
4,6-31+g*,H3O+,-76.6534,-76.6764,-20.3281,898.3417,11.5599
5,6-31+g*,OH-,-75.7850,-75.8046,1.3230,3655.3802,3.3635
6,aug-cc-pvtz,H2O,-76.4412,-76.4626,-8.8199,1627.2502,16.3767
7,aug-cc-pvtz,H3O+,-76.7005,-76.7235,-20.3725,822.9668,42.4613
8,aug-cc-pvtz,OH-,-75.8244,-75.8440,0.9900,3746.2214,6.9548


All frequencies are positive, so every structure is a minimum. Note that $\mathrm{H_3O^+}$ is **pyramidal** ($C_{3v}$), like ammonia, with which it is isoelectronic.

## 3. Proton Affinity and Gas-Phase Acidity

In [5]:
results = []
for basis in basis_sets:
    d = data[data["basis"] == basis].set_index("species")
    PA = (d.loc["H2O", "H (hartree)"] - d.loc["H3O+", "H (hartree)"]) * h2kJ + H_proton
    GB = (d.loc["H2O", "G (hartree)"] - d.loc["H3O+", "G (hartree)"]) * h2kJ + G_proton
    dH_acid = (d.loc["OH-", "H (hartree)"] - d.loc["H2O", "H (hartree)"]) * h2kJ + H_proton
    dG_acid = (d.loc["OH-", "G (hartree)"] - d.loc["H2O", "G (hartree)"]) * h2kJ + G_proton
    results.append({"basis": basis, "PA (kJ/mol)": PA, "GB (kJ/mol)": GB,
                    "ΔH acid (kJ/mol)": dH_acid, "ΔG acid (kJ/mol)": dG_acid})

results.append({"basis": "experiment", "PA (kJ/mol)": 691, "GB (kJ/mol)": 660,
                "ΔH acid (kJ/mol)": 1633, "ΔG acid (kJ/mol)": 1607})
pd.DataFrame(results).round(0)

,basis,PA (kJ/mol),GB (kJ/mol),ΔH acid (kJ/mol),ΔG acid (kJ/mol)
0,6-31g*,707.0,679.0,1777.0,1749.0
1,6-31+g*,678.0,649.0,1615.0,1587.0
2,aug-cc-pvtz,687.0,659.0,1625.0,1598.0
3,experiment,691.0,660.0,1633.0,1607.0


## 4. What the Results Tell Us

- The **proton affinity** changes by up to about 30 kJ/mol between basis sets, but all three land within about 15 kJ/mol of experiment, and aug-cc-pVTZ within a few kJ/mol. The cation holds its electrons tightly, so it is not very demanding.
- The **acidity** is badly overestimated with 6-31G*, by about 140 kJ/mol. Adding diffuse functions (6-31+G*) removes almost all of that error. The extra electron of $\mathrm{OH^-}$ is held loosely and spreads far from the nuclei; without diffuse functions there is nowhere for it to go, so the anion comes out far too high in energy.
- Look at the **HOMO energy** of $\mathrm{OH^-}$ in the table above. It is strongly positive (about +4 eV) with 6-31G* and falls towards zero as diffuse functions are added. A positive HOMO energy means the extra electron is not properly bound. Part of this is the basis set; part is a known weakness of B3LYP for anions (self-interaction error), which is why it stays slightly positive even with aug-cc-pVTZ. The energy **differences** are still good.

```{important} Rule of thumb
Whenever a calculation involves **anions**, lone pairs reaching far from a molecule, or weak interactions, use a basis set with **diffuse functions**: `6-31+G*`, `6-311+G**`, `aug-cc-pVXZ` or `def2-SVPD`/`def2-TZVPD`.
```

These are **gas-phase** numbers. In water, hydration of the ions changes the energetics enormously; the acidity of water in solution ($\mathrm{p}K_a$ 14) corresponds to a free energy of only about 80 kJ/mol. We look at solvation in the *Solvation* notebook.

## Summary
- The first line of the geometry sets the charge and multiplicity.
- Reaction enthalpies need electronic energies **plus** thermal corrections for every species; the proton contributes only $\tfrac{5}{2}RT$.
- Anions require diffuse basis functions.